In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.curve_merger import merge_curves
from src.term_structure.bootstrapping import (
    build_coupon_structure,
    bootstrap_dfs_from_sofr,
    bootstrap_dfs_from_treasury
)
from src.term_structure.curve_interpolator import log_linear_curve_interpolator
from src.term_structure.curve_builder import (
    build_zero_curve,
    build_forward_curve
)

from src.pricing.swap_pricing_engine import (
    par_swap_rate,
    par_swap_curve
)

from src.hedging.hedge_instruments import HedgeInstrument, HedgeUniverse

In [2]:
### Curve Builder
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building synthetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)

# bootstrapping discount factors from sofr curve
df_sofr = bootstrap_dfs_from_sofr(
    sofr_curve = sofr_curve
)

# extracting treasury curve 
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury curve
df_treasury = bootstrap_dfs_from_treasury(
    treasury_curve = treasury_curve,
    short_dfs = df_sofr
)

# concatenating short-end and long-end into a single curve -> full curve
df_full_curve = merge_curves(
    short_curve = df_sofr,
    long_curve = df_treasury
)

# constructing semi-annual coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

# log-linear discount curve interpolation
df_loglinear_interp = log_linear_curve_interpolator(
    df_curve = df_full_curve,
    _target_times = coupon_structure
)

# building zero curve from log-linear discount curve
zero_curve_full = build_zero_curve(
    df_curve = df_loglinear_interp
)

# building forward curve from log-linear discount curve
forward_curve_full = build_forward_curve(
    df_curve = df_loglinear_interp
)

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


In [ ]:
### Hedge Instruments
# extracting latest par swap curve
latest_par_swap_curve = par_swap_curve(
    df_curve = df_loglinear_interp,
    maturities= [2.0, 5.0, 10.0, 30.0],
    freq = 2
).iloc[-1]

# creating hedge instruments
hedges = [
    HedgeInstrument('HedgeSwap_2Y', 2.0, fixed_rate = float(latest_par_swap_curve.loc[2])),
    HedgeInstrument('HedgeSwap_5Y', 5.0, fixed_rate = float(latest_par_swap_curve.loc[5])),
    HedgeInstrument('HedgeSwap_10Y', 10.0, fixed_rate = float(latest_par_swap_curve.loc[10])),
    HedgeInstrument('HedgeSwap_30Y', 30.0, fixed_rate = float(latest_par_swap_curve.loc[30]))
]

# hedge universe
hedge_universe = HedgeUniverse(
    instruments = hedges
)

# building hedge swaps
hedge_swaps = hedge_universe.build_IRswaps()

hedge_universe.summary()

,Instrument,Type,Maturity,Par Rate,Notional,CouponFreq
0,HedgeSwap_2Y,receiver,2.0,0.0378,1000000,2
1,HedgeSwap_5Y,receiver,5.0,0.0391,1000000,2
2,HedgeSwap_10Y,receiver,10.0,0.0430,1000000,2
3,HedgeSwap_30Y,receiver,30.0,0.0489,1000000,2
